# Decision Consistency Evaluation — Intel Image Classification (Scene Recognition)

**Dataset:** Intel Image Classification — 6 natural scene classes (buildings, forest, glacier, mountain, sea, street), 150×150 real-world photography.

This is the strongest cross-domain test: models are ImageNet-pretrained and evaluated on a completely different scene-recognition task. The methodology is identical to CIFAR and STL-10, making results directly comparable. The cross-domain setting lets us isolate the CAG diagnostic — models that score high on Consistency Score but low on accuracy reveal the 'Consistently Wrong' failure mode.

Expected folder layout:
```
seg_test/seg_test/buildings/*.jpg
seg_test/seg_test/forest/*.jpg
seg_test/seg_test/glacier/*.jpg
seg_test/seg_test/mountain/*.jpg
seg_test/seg_test/sea/*.jpg
seg_test/seg_test/street/*.jpg
```

**Update paths in the Configuration cell before running.**

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
INTEL_TEST_DIR = r'C:\Users\Acer\Downloads\Intel Image Classification\seg_test\seg_test'
PREV_CIFAR_CSV = r'E:\decision_consistency_cifar_outputs\tables\summary_table.csv'
PREV_STL10_CSV = r'E:\decision_consistency_stl10_outputs\tables\summary_table_stl10.csv'
OUTPUT_DIR     = r'E:\decision_consistency_intel_outputs'

NUM_SAMPLES = 1000
SEED        = 42
ALPHA       = 0.5

In [ ]:
import os, json, random, io, csv, gc
from collections import defaultdict
from pathlib import Path

import numpy as np
from scipy.stats import wilcoxon, pearsonr, spearmanr

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import torch
import torch.nn.functional as F
import torchvision.transforms.functional as TF
from torchvision import models, transforms, datasets
from PIL import Image
from tqdm import tqdm

import warnings
warnings.filterwarnings('ignore')
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:128'

for sub in ['tables','figures/consistency','figures/gradcam',
            'figures/perclass','figures/ablation','figures/correlation']:
    os.makedirs(os.path.join(OUTPUT_DIR, sub), exist_ok=True)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

In [ ]:
# Load Intel dataset — ImageFolder auto-discovers class sub-folders
raw_transform = transforms.Compose([
    transforms.Resize((150, 150)),
    transforms.ToTensor()
])
intel_dataset = datasets.ImageFolder(root=INTEL_TEST_DIR, transform=raw_transform)
INTEL_CLASSES = intel_dataset.classes
print('Classes:', INTEL_CLASSES, '| Total images:', len(intel_dataset))

all_indices = list(range(len(intel_dataset)))
random.seed(SEED); random.shuffle(all_indices)
selected = all_indices[:NUM_SAMPLES]

X_intel, y_intel = [], []
for idx in tqdm(selected, desc='Loading Intel images'):
    img_tensor, label = intel_dataset[idx]
    X_intel.append(img_tensor); y_intel.append(label)
y_intel = np.array(y_intel)
print(f'Loaded {len(X_intel)} images')
for ci, cn in enumerate(INTEL_CLASSES):
    print(f'  {cn:12s}: {(y_intel==ci).sum()}')

In [ ]:
def load_models(device):
    zoo = [('ResNet-18', models.resnet18), ('ResNet-50', models.resnet50),
           ('VGG-16', models.vgg16), ('MobileNetV2', models.mobilenet_v2)]
    md = {}
    for name, fn in zoo:
        print(f'Loading {name}...', end=' ', flush=True)
        md[name] = fn(pretrained=True).eval().to(device); print('OK')
    return md

MODELS = load_models(device)
model_names = list(MODELS.keys())

In [ ]:
preprocess = transforms.Compose([
    transforms.Resize((224, 224)), transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])

def prepare(t):
    return preprocess(transforms.ToPILImage()(t.cpu().clamp(0,1))).unsqueeze(0).to(device)

def apply_jpeg(t, q=75):
    buf = io.BytesIO()
    transforms.ToPILImage()(t.cpu().clamp(0,1)).save(buf, format='JPEG', quality=q)
    buf.seek(0); return transforms.ToTensor()(Image.open(buf))

def generate_perturbations(t):
    noise = 0.01 * torch.randn_like(t)
    return {'rotation':   TF.rotate(t, angle=2),
            'brightness': TF.adjust_brightness(t, brightness_factor=1.1),
            'noise':      torch.clamp(t + noise, 0.0, 1.0),
            'contrast':   TF.adjust_contrast(t, contrast_factor=1.1),
            'blur':       TF.gaussian_blur(t, kernel_size=3, sigma=0.5),
            'jpeg':       apply_jpeg(t)}

def infer(model, t):
    with torch.no_grad():
        probs = F.softmax(model(prepare(t)), dim=1)
    return probs.argmax(dim=1).item(), probs.max().item()

def compute_metrics(orig_pred, orig_conf, pert_results, alpha=ALPHA):
    K = len(pert_results)
    S = sum(1 for p,_ in pert_results.values() if p == orig_pred) / K
    D = sum(abs(c-orig_conf)/max(orig_conf,1-orig_conf) for _,c in pert_results.values()) / K
    return {'Label_Stability':S, 'Confidence_Deviation':D,
            'Consistency_Score': alpha*S + (1-alpha)*(1-D)}

print('Pipeline ready.')

In [ ]:
results = {'Intel-Scene': {}}

for model_name, model in MODELS.items():
    per_sample = []
    for img, true_label in tqdm(zip(X_intel, y_intel),
                                total=len(X_intel),
                                desc=f'Intel-Scene/{model_name}'):
        orig_pred, orig_conf = infer(model, img)
        pert_results = {k: infer(model, v) for k, v in generate_perturbations(img).items()}
        m = compute_metrics(orig_pred, orig_conf, pert_results)
        m['true_label'] = int(true_label)
        m['class_name'] = INTEL_CLASSES[int(true_label)]
        m['orig_pred']  = orig_pred
        m['orig_conf']  = orig_conf
        per_sample.append(m)
    results['Intel-Scene'][model_name] = per_sample
    ls = np.mean([m['Label_Stability']     for m in per_sample])
    cd = np.mean([m['Confidence_Deviation'] for m in per_sample])
    cs = np.mean([m['Consistency_Score']    for m in per_sample])
    print(f'  [Intel-Scene][{model_name}]  LS={ls:.3f}  CD={cd:.3f}  CS={cs:.3f}')

print('Evaluation complete.')

In [ ]:
# Summary + merged cross-dataset table
header = ['Dataset','Model','Avg_Label_Stability','Avg_Confidence_Deviation',
          'Avg_Consistency_Score','Std_Consistency_Score']
rows = []
print(f'{"Model":<14} {"LS":>6} {"CD":>7} {"CS":>7} {"±CS":>7}')
print('-' * 44)
for mn in model_names:
    ms  = results['Intel-Scene'][mn]
    ls  = np.mean([m['Label_Stability']      for m in ms])
    cd  = np.mean([m['Confidence_Deviation']  for m in ms])
    cs  = np.mean([m['Consistency_Score']     for m in ms])
    std = np.std( [m['Consistency_Score']     for m in ms])
    print(f'{mn:<14} {ls:6.3f} {cd:7.3f} {cs:7.3f} {std:7.3f}')
    rows.append({'Dataset':'Intel-Scene','Model':mn,
                 'Avg_Label_Stability':round(ls,4),
                 'Avg_Confidence_Deviation':round(cd,4),
                 'Avg_Consistency_Score':round(cs,4),
                 'Std_Consistency_Score':round(std,4)})

csv_path = os.path.join(OUTPUT_DIR, 'tables', 'summary_table_intel.csv')
with open(csv_path, 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=header); w.writeheader(); w.writerows(rows)
print('Saved:', csv_path)

# Merge with previous datasets
all_rows = list(rows)
for prev_csv in [PREV_CIFAR_CSV, PREV_STL10_CSV]:
    if os.path.exists(prev_csv):
        with open(prev_csv) as f:
            all_rows.extend(csv.DictReader(f))
        print('Merged:', prev_csv)
master_path = os.path.join(OUTPUT_DIR, 'tables', 'MASTER_all_datasets.csv')
with open(master_path, 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=header); w.writeheader(); w.writerows(all_rows)
print('Master table saved:', master_path)

In [ ]:
# α-ablation
alpha_values = [0.3, 0.4, 0.5, 0.6, 0.7]
fig, ax = plt.subplots(figsize=(8, 4))
for mn, marker in zip(model_names, ['o','s','^','D']):
    ms   = results['Intel-Scene'][mn]
    ls_a = np.array([m['Label_Stability']     for m in ms])
    cd_a = np.array([m['Confidence_Deviation'] for m in ms])
    vals = [float(np.mean(a*ls_a + (1-a)*(1-cd_a))) for a in alpha_values]
    ax.plot(alpha_values, vals, marker=marker, label=mn, linewidth=2, markersize=7)
ax.set_xlabel('α', fontsize=11)
ax.set_ylabel('Average Consistency Score', fontsize=11)
ax.set_title('α-Ablation Study — Intel Image Classification\n'
             'Model rankings stable across all α', fontsize=11)
ax.set_xticks(alpha_values); ax.set_ylim(0, 1)
ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout()
path = os.path.join(OUTPUT_DIR, 'figures', 'ablation', 'alpha_ablation_intel.png')
plt.savefig(path, dpi=300, bbox_inches='tight'); plt.close()
print('Saved:', path)

In [ ]:
# Cross-model bar chart
avg_ls = [np.mean([m['Label_Stability']      for m in results['Intel-Scene'][mn]]) for mn in model_names]
avg_cs = [np.mean([m['Consistency_Score']     for m in results['Intel-Scene'][mn]]) for mn in model_names]
avg_cd = [np.mean([m['Confidence_Deviation']  for m in results['Intel-Scene'][mn]]) for mn in model_names]
x, w = np.arange(len(model_names)), 0.25
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(x-w, avg_ls, w, label='Label Stability',      color='steelblue')
ax.bar(x,   avg_cs, w, label='Consistency Score',    color='darkorange')
ax.bar(x+w, avg_cd, w, label='Confidence Deviation', color='seagreen')
ax.set_xticks(x); ax.set_xticklabels(model_names, fontsize=11)
ax.set_ylim(0,1); ax.set_ylabel('Score')
ax.set_title('Cross-Model Decision Consistency — Intel Image Classification', fontsize=12)
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
path = os.path.join(OUTPUT_DIR, 'figures', 'consistency', 'crossmodel_Intel_Scene.png')
plt.savefig(path, dpi=300, bbox_inches='tight'); plt.close()
print('Saved:', path)

In [ ]:
# Per-class consistency
for mn in model_names:
    ms           = results['Intel-Scene'][mn]
    class_scores = defaultdict(list)
    for m in ms: class_scores[m['class_name']].append(m['Consistency_Score'])
    sorted_cls = sorted({cls: np.mean(s) for cls,s in class_scores.items()}.items(),
                        key=lambda x: x[1])
    cls_names = [c for c,_ in sorted_cls]
    cls_vals  = [v for _,v in sorted_cls]
    fig, ax   = plt.subplots(figsize=(8, 4))
    ax.bar(cls_names, cls_vals,
           color=['tomato' if v<0.5 else 'steelblue' for v in cls_vals],
           edgecolor='white')
    ax.set_ylim(0,1); ax.set_ylabel('Avg Consistency Score')
    ax.set_title(f'Per-Class Consistency — Intel-Scene / {mn}', fontsize=10)
    ax.axhline(0.5, color='gray', linestyle='--', lw=1)
    plt.tight_layout()
    fname = f'perclass_Intel_{mn}.png'.replace('-','_').replace(' ','_')
    plt.savefig(os.path.join(OUTPUT_DIR, 'figures', 'perclass', fname),
                dpi=300, bbox_inches='tight'); plt.close()
    print('Saved:', fname)

In [ ]:
# Significance tests
cs_arrays = {mn: np.array([m['Consistency_Score'] for m in results['Intel-Scene'][mn]])
             for mn in model_names}
print('=== Wilcoxon Tests (Intel-Scene) ===')
print(f'{"Pair":<28} {"W":>10} {"p":>10} {"sig":>6}')
print('-' * 58)
sig_rows = []
for i, mn1 in enumerate(model_names):
    for mn2 in model_names[i+1:]:
        w, p = wilcoxon(cs_arrays[mn1], cs_arrays[mn2])
        pair = f'{mn1} vs {mn2}'
        print(f'{pair:<28} {w:>10.2f} {p:>10.4f} {"***" if p<0.05 else "ns":>6}')
        sig_rows.append({'Pair':pair,'W':round(w,2),'p_value':round(p,6),'Significant':p<0.05})
csv_path = os.path.join(OUTPUT_DIR, 'tables', 'wilcoxon_tests_intel.csv')
with open(csv_path, 'w', newline='') as f:
    dw = csv.DictWriter(f, fieldnames=['Pair','W','p_value','Significant'])
    dw.writeheader(); dw.writerows(sig_rows)
print('Saved:', csv_path)